In [ ]:
import pandas as pd
import numpy as np
import random
import warnings
from copy import deepcopy
warnings.filterwarnings('ignore')

## Wczytanie danych

In [ ]:
data = pd.read_csv("allegro-api-transactions.csv")
data.head(4)

In [ ]:
data.info()

## Kodowanie zmiennych kategorycznych

In [ ]:
import category_encoders as ce

X = data.drop(columns = ["price"])
y = data["price"]

### Target Encoding

In [ ]:
encoder = ce.TargetEncoder(cols=["it_location"])
it_location_1 = encoder.fit(X, y).transform(X)["it_location"]
it_location_1

Oczywist? przewag? target encoding nad one-hot jest wielko?? jak? zajmuje. Zwr??my uwag?, ?e jedn? kolumn? zast?pujemy tak?e dok?adnie jedn?. Mo?emy sprawdzi?, ile unikatowych warto?ci zawiera kolumna it_location, aby u?wiadomi? sobie, ile nowych by?my musieli posiada? gdyby?my chcieli zastosowa? tutaj one-hot encoding:

In [ ]:
len(data["it_location"].value_counts())

Raczej nie jeste?my w stanie efektywnie operowa? na ramce danych zawieraj?cej tak? liczb? kolumn, a szczeg?lnie, ?e it_location to nie jedyna kolumna, kt?r? nale?y zakodowa?. Oczywi?cie je?li si? dobrze przyjrzymy kolumnie it_location, to zawiera ona na przyk?ad "Warszawa" zapisane tak?e jako "WARSZAWA" czy "warszawa" - wszystko to by mo?na by?o porz?dnie przearan?owa?, tak?e u?y? znacznika "Other" dla rzadko wyst?puj?cych warto?ci i wtedy dopiero my?le? o kodowaniu poprzez one-hot encoding (bowiem jest on na pewno dok?adniejszy, a korzystaj?c z target encoding tracimy wiele informacji w kolumnie). Mimo to, nawet wtedy zajmuje on (one-hot) znacznie wi?ksze miejsce w pami?ci i mo?e spowodowa? wolniejsze obliczenia.

### One-hot encoding

Teraz zastosujemy metod? one-hot encoding, ale dla kolumny, kt?ra posiada o wiele mniej unikatowych warto?ci - main_category.

In [ ]:
encoder = ce.OneHotEncoder(cols=["main_category"])
df_1 = encoder.fit(X, y).transform(X)
df_1.columns

Widzimy, ?e korzystaj?c z one-hot encodingu otrzymujemy 27 nowych kolumn - jest to dosy? du?a, ale jeszcze rozs?dna liczba (szczeg?lnie, ?e nie tracimy ?adnych informacji w por?wnaniu do pocz?tkowych danych).

### Count encoding

In [ ]:
encoder = ce.CountEncoder(cols=["main_category"])
df_2 = encoder.fit(X, y).transform(X)
df_2["main_category"]

Count encoder dzia?a na zasadzie zamienienia ka?dej warto?ci liczb? jej wyst?pie? w ca?ej kolumnie, tzn. je?li jaka? kategoria by?a w kolumnie 10000 razy, to po zakodowaniu b?dziemy mieli zamiast nazwy tej kategorii, w?a?nie tak? liczb?. Taki spos?b kodowania z zalet na pewno nie rozszerza nam ramki, ale aby mia?o ono sens musz? by? odpowiednie do tego dane, wydaje si? jakby w naszym przypadku takie kodowanie nie mia?o zbyt du?ego sensu.

### Leave One Out encoding

In [ ]:
encoder = ce.LeaveOneOutEncoder(cols=["main_category"])
df_3 = encoder.fit(X, y).transform(X)
df_3["main_category"]

Leave One Out Encoder dzia?a w podobny spos?b do Target Encoder, ale tak jak nazwa wskazuje, w obliczaniu warto?ci pomija target w danym wierszu.

## Uzupe?nianie brak?w

In [ ]:
random.seed(131)
# ograniczymy nasz? ramk? do 10000 wierszy w celu przyspieszenia oblicze?
data = data[["price", "it_seller_rating", "it_quantity"]].sample(10000).reset_index(drop=True)

### Nearest Neighbors Imputation

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error

rmse = np.array([])

for i in range(10):
    df = deepcopy(data)
    n = df.shape[0]
    random_numbers = random.sample(list(df.index.values), k = n // 10)
    df.loc[random_numbers, "it_seller_rating"] = np.nan
    imputer = KNNImputer(n_neighbors = 10, weights = "uniform")
    temp = imputer.fit_transform(df)
    df = pd.DataFrame(temp, columns = data.columns)
    y_true = data.loc[random_numbers, "it_seller_rating"]
    y_pred = df.loc[random_numbers, "it_seller_rating"]
    rmse = np.append(rmse, np.sqrt(mean_squared_error(y_true, y_pred)))

Po wykonaniu eksperymentu 10-krotnie mo?emy wy?wietli? otrzymane warto?ci RMSE:

In [ ]:
print(rmse)

In [ ]:
rmse.std()

Widzimy, ?e odchylenie standardowe 10 test?w wynosi ok. 2940, czyli dosy? du?o.
Powt?rzymy jeszcze eksperyment, ale usuwaj?c warto?ci z dw?ch kolumn.

In [ ]:
rmse2_SR = np.array([])
rmse2_Q = np.array([])

for i in range(10):
    df = deepcopy(data)
    n = df.shape[0]
    random_numbers_1 = random.sample(list(df.index.values), k = n // 10)
    random_numbers_2 = random.sample(list(df.index.values), k = n // 10)
    df.loc[random_numbers_1, "it_seller_rating"] = np.nan
    df.loc[random_numbers_2, "it_quantity"] = np.nan
    imputer = KNNImputer(n_neighbors = 10, weights = "uniform")
    temp = imputer.fit_transform(df)
    df = pd.DataFrame(temp, columns = data.columns)
    y_true_SR = data.loc[random_numbers_1, "it_seller_rating"]
    y_pred_SR = df.loc[random_numbers_1, "it_seller_rating"]
    rmse2_SR = np.append(rmse2_SR, np.sqrt(mean_squared_error(y_true_SR, y_pred_SR)))
    y_true_Q = data.loc[random_numbers_1, "it_quantity"]
    y_pred_Q = df.loc[random_numbers_1, "it_quantity"]
    rmse2_Q = np.append(rmse2_Q, np.sqrt(mean_squared_error(y_true_Q, y_pred_Q)))

Mo?emy ponownie sprawdzi? warto?ci RMSE oraz odchylenie standardowe tych warto?ci dla obu kolumn.

In [ ]:
print(rmse2_SR)

In [ ]:
rmse2_SR.std()

In [ ]:
print(rmse2_Q)

In [ ]:
rmse2_Q.std()

Aby zobaczy? lepsze por?wnanie otrzymanych warto?ci z obu eksperyment?w stworzymy boxploty.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11,6))
bp = ax.boxplot([rmse, rmse2_SR])
plt.show()

### Podsumowanie

Przy ustawionym ziarnie losowo?ci otrzymujemy, ?e warto?ci RMSE, gdy usuwali?my warto?ci tylko z jednej kolumny s? przeci?tnie ni?sze (mamy ni?sz? warto?? minimaln?, maksymaln? i wszystkie kwartyle) - czyli lepiej. Jest to zgodne z intuicj?, ?e przy wi?kszych brakach w danych u?ywane narz?dzie b?dzie zachowywa?o si? gorzej, ale przy ekspermentowaniu z ziarnem losowo?ci tak naprawd? otrzymywa?em wyniki raz takie, a raz takie, ?e w drugim eksperymencie braki zosta?y dopasowane lepiej (wi?c ci??ko oceni?, czy braki w drugiej kolumnie realnie bardzo wp?ywaj? na zachowanie narz?dzia).

Podsumowuj?c, u?ywane narz?dzie raczej nie spisa?o si? dobrze, wi?c w takim przypadku jego u?ywanie jest s?abym pomys?em (zreszt? nic dziwnego, ci??ko ustali? "it_seller_rating" na podstawie nabardziej podobnych wierszy, gdy mamy dane kolumny tylko "price" i "it_quantity").